In [1]:
import dynamiqs as dq
import jax
import jax.numpy as jnp
from jax import value_and_grad,grad
import qcsys as qs
import jaxquantum as jqt
jax.config.update('jax_platform_name', 'cpu')
dq.set_device('cpu')

import optax
import warnings
warnings.filterwarnings("ignore")
from IPython.display import clear_output


2024-03-25 01:06:09.949964: E external/xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: UNKNOWN ERROR (100)


### let's first optimize the transmon frequency to be fluxonium 0-7 transition

In [2]:
Ec_t = 0.2
def objective(params):
    Ej_t = params[0]
    qst = qs.SingleChargeTransmon.create(
        N = 4,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=30
    )
    return abs(qst.eig_systems["vals"][1] - qst.eig_systems["vals"][0] - 7.182920828753576)


optimizer = optax.adam(learning_rate=0.1) 
params = jnp.array([40.0])
opt_state = optimizer.init(params)

num_steps = 1000
for step in range(num_steps):
    val, grads = jax.value_and_grad(objective)(params)
    if step %10 == 0:
        clear_output()
        print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
    if val < 1e-6:
        clear_output()
        print(f"iter: {step}, val={val:.4f} grads: {grads}, params: {params}")
        break
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

print(f'Optimized params: {params}')

iter: 162, val=0.0000 grads: [-0.10836512], params: [34.12111946]
Optimized params: [34.12111946]


In [ ]:
def calculate_eig(H: jqt.Qarray):
    vals, kets = jnp.linalg.eigh(H.data)
    return (vals,kets) # Here kets is equivalent to the S in qutip.Qobj.transform

def transform_op_into_dressed_basis_jax(op_matrix: jqt.Qarray, 
                                        S: jax.Array) -> jax.Array:
    """
    Transform an operator into the dressed basis using JAX.

    Parameters:
    - op_matrix: A 2D JAX array representing the operator's matrix.
    - S: A 2D JAX array representing the dressed eigenvectors similar to the S in qutip.Qobj.transform

    Returns:
    - A 2D JAX array representing the transformed operator.
    """
    data = jnp.dot(S, jnp.dot(op_matrix.data, S.T.conj()))
    return data

def get_hamiltonian_and_operators(
        Ej_f = 2.7,
        Ec_f = 0.6,
        El_f = 0.13,
        Ej_t = 40
    ):
    qsf = qs.Fluxonium.create(
        20,
        {"Ej": Ej_f, "Ec": Ec_f, "El": El_f, "phi_ext": 0.0},
        N_pre_diag=100,
        use_linear = False
    )

    Ec_t = 0.5
    qst_sc = qs.SingleChargeTransmon.create(
        N = 4,
        params = {"Ej": Ej_t, "Ec": Ec_t,"ng":0.0},
        N_max_charge=30
    )
    devices = [qsf, qst_sc]
    f_indx = 0
    t_indx = 1
    Ns = [device.N for device in devices]
    tn = qs.promote(qst_sc.ops['n'], t_indx, Ns)
    fn = qs.promote(qsf.ops["n"], f_indx, Ns)
    g_tf = 0.2
    system = qs.System.create(devices, couplings=[
        g_tf *  fn @ tn
    ])
    system.params["g_tf"] = g_tf
    Es, kets = system.calculate_eig_linear()

    driven_op = transform_op_into_dressed_basis_jax(tn, calculate_eig(system.get_H())[1].T)
    w_d = 
    return driven_op, w_d
def objective(params):
    amp_with_2pi = params[0]
    pulse_length = params[1]

    
    t_rise = 30
    t_tot = t_rise + pulse_length

    tlist = jnp.linspace(0,t_tot,int(t_tot))

    pulse_shape_args={
        'w_d': w_d,
        'amp': amp_with_2pi/(2*np.pi),
        't_rise': t_rise,
        't_square': pulse_length - t_rise
    }          

    def _H(t):
        _H =  jnp.array(system.diag_dressed_hamiltonian)
        _H += jnp.array(driven_op)* pulse_shape_func(t, pulse_shape_args)
        return _H 
    H =  timecallable(_H)

    result0 = dq.sesolve(
                H = H,
                psi0 = jnp.array(system.truncate_function(qutip.basis(system.hilbertspace.dimension, system.product_to_dressed[(0,0)]))),
                tsave = tlist,
                solver = dq.solver.Tsit5(
                        rtol= 1e-06,
                        atol= 1e-06,
                        safety_factor= 0.9,
                        min_factor= 0.2,
                        max_factor = 5.0,
                        max_steps = int(1e4*(tlist[-1]-tlist[0])),
                    )
                )
    result1 = dq.sesolve(
                H = H,
                psi0 = jnp.array(system.truncate_function(qutip.basis(system.hilbertspace.dimension, system.product_to_dressed[(1,0)]))),
                tsave = tlist,
                solver = dq.solver.Tsit5(
                        rtol= 1e-06,
                        atol= 1e-06,
                        safety_factor= 0.9,
                        min_factor= 0.2,
                        max_factor = 5.0,
                        max_steps = int(1e4*(tlist[-1]-tlist[0])),
                    )
                )
    result2 = dq.sesolve(
                H = H,
                psi0 =jnp.array(system.truncate_function(qutip.basis(system.hilbertspace.dimension, system.product_to_dressed[(2,0)]))),
                tsave = tlist,
                solver = dq.solver.Tsit5(
                        rtol= 1e-06,
                        atol= 1e-06,
                        safety_factor= 0.9,
                        min_factor= 0.2,
                        max_factor = 5.0,
                        max_steps = int(1e4*(tlist[-1]-tlist[0])),
                    )
                )
    return 1 - dq.expect(qutip.ket2dm(system.truncate_function(qutip.basis(system.hilbertspace.dimension, system.product_to_dressed[(0,1)]))), result0.states[-1]).real +\
          dq.expect(qutip.ket2dm(system.truncate_function(qutip.basis(system.hilbertspace.dimension, system.product_to_dressed[(1,1)]))), result1.states[-1]).real +\
              dq.expect(qutip.ket2dm(system.truncate_function(qutip.basis(system.hilbertspace.dimension, system.product_to_dressed[(2,1)]))), result2.states[-1]).real




In [ ]:

amp_with_2pi = 0.011026707187734986
pulse_length = 308.37376398206834-6
flux = 0.40367803977430805

optimizer = optax.adam(learning_rate=0.01) 
params = jnp.array([amp_with_2pi, pulse_length, flux])
opt_state = optimizer.init(params)

num_steps = 1000



for step in range(num_steps):
    population_value, grads = jax.value_and_grad (objective)(params)
    print(f"Population={population_value:.4f} grads: {grads}")
    
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    amp_with_2pi, pulse_length = params


print(f'Optimized amp_with_2pi: {amp_with_2pi:.2f}')
print(f'Optimized pulse_length: {pulse_length:.2f}')